# DIA scraper — estructura común final

Flujo:

1. **Scraping** de las 20 búsquedas, con scroll interno y reintentos automáticos.
2. **RAW:** todas las apariciones se unen y se guardan en un único CSV.
3. **Procesado:** deduplicación por `product_id`, normalización de unidades y eliminación de filas sin precio o unidad comparable.
4. **PROCESSED:** un único CSV final limpio.

Archivos:
- `data/raw/dia_products_raw.csv`
- `data/processed/dia_products.csv`

In [ ]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

HOME_URL = "https://www.dia.es/"
CARD_SELECTOR = 'div[data-test-id="product-card"]'
SCROLL_SELECTOR = ".search-component__content"

WAIT = 20
MAX_BATCHES = 40
NEXT_BATCH_TIMEOUT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

In [ ]:
current = Path.cwd().resolve()
BASE_DIR = current.parent if current.name == "scrapers" else current

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "dia_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "dia_products.csv"

print("RAW:", RAW_PATH)
print("PROCESSED:", PROCESSED_PATH)

In [ ]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies_if_needed(driver):
    try:
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(., 'Aceptar')]")
            )
        )
        button.click()
        print("Cookies aceptadas.")
    except TimeoutException:
        pass


def go_to_home(driver):
    driver.get(HOME_URL)
    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(1)
    accept_cookies_if_needed(driver)


def assert_no_security_challenge(driver):
    text = driver.find_element(By.TAG_NAME, "body").text.lower()
    messages = [
        "confirma que no eres un robot",
        "comprueba que eres humano",
        "access denied",
        "captcha",
    ]
    if any(message in text for message in messages):
        raise RuntimeError("Verificación de seguridad detectada.")

In [ ]:
def find_search_box(driver):
    selectors = [
        'input[type="search"]',
        'input[data-test-id*="search"]',
        'input[data-testid*="search"]',
        'input[placeholder*="Buscar"]',
        'input[placeholder*="buscar"]',
        'input[aria-label*="Buscar"]',
        'input[aria-label*="buscar"]',
    ]

    def visible_candidate(_):
        for selector in selectors:
            for element in driver.find_elements(By.CSS_SELECTOR, selector):
                if element.is_displayed() and element.is_enabled():
                    return element
        return False

    return WebDriverWait(driver, WAIT).until(visible_candidate)


def search_dia(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SCROLL_SELECTOR))
    )

    WebDriverWait(driver, WAIT).until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)) > 0
    )

    assert_no_security_challenge(driver)
    time.sleep(1)

In [ ]:
def load_all_cards(driver):
    container = driver.find_element(By.CSS_SELECTOR, SCROLL_SELECTOR)
    count = len(driver.find_elements(By.CSS_SELECTOR, CARD_SELECTOR))
    print("Tarjetas iniciales:", count)

    for batch in range(1, MAX_BATCHES + 1):
        previous = count

        state = driver.execute_script(
            '''
            const el = arguments[0];
            return {
                scrollHeight: el.scrollHeight,
                clientHeight: el.clientHeight
            };
            ''',
            container
        )

        target = max(
            0,
            state["scrollHeight"] - state["clientHeight"] - 450
        )

        driver.execute_script(
            "arguments[0].scrollTop = arguments[1];",
            container,
            target
        )

        try:
            WebDriverWait(
                driver,
                NEXT_BATCH_TIMEOUT,
                poll_frequency=0.3
            ).until(
                lambda d: len(
                    d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)
                ) > previous
            )

            count = len(
                driver.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)
            )

            print(f"Tanda {batch}: {previous} -> {count}")

        except TimeoutException:
            return previous, "complete"

    return count, "max_batches_reached"

In [ ]:
def clean_price(text):
    if not text:
        return None

    match = re.search(
        r"\d+(?:[.,]\d+)?",
        str(text).replace("\xa0", " ")
    )

    return (
        float(match.group().replace(",", "."))
        if match
        else None
    )


def extract_price_per_unit(text):
    if not text:
        return None, None

    value = clean_price(text)

    match = re.search(
        r"€\s*/\s*([A-Za-zÁÉÍÓÚáéíóú]+)",
        text
    )

    unit = match.group(1).upper() if match else None
    return value, unit


def extract_products(driver, searched_item):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select(CARD_SELECTOR)

    rows = []
    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        link = card.select_one(
            '[data-test-id="search-product-card-name"]'
        )
        price_el = card.select_one(
            '[data-test-id="search-product-card-unit-price"]'
        )
        ppu_el = card.select_one(
            '[data-test-id="search-product-card-kilo-price"]'
        )

        if not link or not price_el:
            continue

        price_raw = price_el.get_text(" ", strip=True)
        ppu_raw = (
            ppu_el.get_text(" ", strip=True)
            if ppu_el
            else None
        )

        ppu, unit = extract_price_per_unit(ppu_raw)

        rows.append({
            "searched_item": searched_item,
            "supermarket": "DIA",
            "product_id": card.get("object_id"),
            "name": link.get_text(" ", strip=True),
            "price_raw": price_raw,
            "price": clean_price(price_raw),
            "price_per_unit_raw": ppu_raw,
            "price_per_unit": ppu,
            "unit": unit,
            "url": link.get("href"),
            "scraped_at": scraped_at,
        })

    return pd.DataFrame(rows)

In [ ]:
driver = start_driver()
time.sleep(2)
accept_cookies_if_needed(driver)

all_results = []
summary = []

for i, term in enumerate(SEARCH_TERMS, start=1):
    success = False
    started = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 2):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(SEARCH_TERMS)}] {term} · intento {attempt}")

        try:
            go_to_home(driver)
            search_dia(driver, term)

            loaded, load_status = load_all_cards(driver)
            df_term = extract_products(driver, term)
            extracted = len(df_term)

            if load_status != "complete":
                raise RuntimeError(
                    f"La carga terminó con estado '{load_status}'."
                )

            if extracted == 0 or extracted != loaded:
                raise RuntimeError(
                    f"Tarjetas cargadas {loaded}, filas extraídas {extracted}."
                )

            all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "cards_loaded": loaded,
                "rows_extracted": extracted,
                "attempts": attempt,
                "status": "OK",
                "seconds": round(time.monotonic() - started, 2),
            })

            print("Estado: OK")
            success = True
            break

        except Exception as exc:
            print(f"Intento {attempt} fallido: {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES + 1:
                time.sleep(2)

    if not success:
        summary.append({
            "searched_item": term,
            "cards_loaded": None,
            "rows_extracted": 0,
            "attempts": MAX_RETRIES + 1,
            "status": "ERROR",
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)

## RAW — unir todas las búsquedas y guardar un único archivo

In [ ]:
if not all_results:
    raise RuntimeError("No se extrajo ninguna búsqueda.")

df_dia_raw = pd.concat(all_results, ignore_index=True)

df_dia_raw.to_csv(
    RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Filas RAW:", len(df_dia_raw))
print("IDs distintos:", df_dia_raw["product_id"].nunique())
print("RAW guardado:", RAW_PATH)

## PROCESSED — deduplicar, normalizar y eliminar filas incompletas

In [ ]:
def normalize_dia_row(row):
    raw = row["price_per_unit_raw"]

    if pd.isna(raw):
        return None, None

    text = str(raw).upper().replace(",", ".")
    value = row["price_per_unit"]

    if pd.isna(value):
        value = clean_price(raw)

    if value is None or pd.isna(value):
        return None, None

    if re.search(r"/\s*100\s*ML\.?", text):
        return float(value) * 10, "LITRO"

    if re.search(r"/\s*100\s*(?:GR|G)\.?", text):
        return float(value) * 10, "KILO"

    unit = row["unit"]

    mapping = {
        "KG": "KILO",
        "KILO": "KILO",
        "L": "LITRO",
        "LITRO": "LITRO",
        "UNIDAD": "UNIDAD",
        "DOCENA": "DOCENA",
        "LAVADO": "LAVADO",
    }

    normalized_unit = mapping.get(
        str(unit).upper()
        if pd.notna(unit)
        else None
    )

    return float(value), normalized_unit

In [ ]:
df_dia_processed = (
    df_dia_raw[
        df_dia_raw["product_id"].notna()
    ]
    .drop_duplicates(subset=["product_id"])
    .reset_index(drop=True)
    .copy()
)

normalized = df_dia_processed.apply(
    normalize_dia_row,
    axis=1
)

df_dia_processed["price_per_unit"] = normalized.apply(lambda x: x[0])
df_dia_processed["unit"] = normalized.apply(lambda x: x[1])

df_dia_processed["price_per_unit"] = pd.to_numeric(
    df_dia_processed["price_per_unit"],
    errors="coerce"
).round(2)

before_drop = len(df_dia_processed)

df_dia_processed = (
    df_dia_processed[
        df_dia_processed["price"].notna()
        & df_dia_processed["price_per_unit"].notna()
        & df_dia_processed["unit"].notna()
    ]
    .reset_index(drop=True)
)

print("Únicos antes de eliminar incompletos:", before_drop)
print("Productos finales:", len(df_dia_processed))
print("Eliminados por precio/unidad incompletos:", before_drop - len(df_dia_processed))
print("\nUnidades:")
print(df_dia_processed["unit"].value_counts(dropna=False))

In [ ]:
df_dia_processed.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("PROCESSED guardado:", PROCESSED_PATH)

In [ ]:
driver.quit()
print("Driver cerrado.")